# Summit 2026 HOL — AI Security & Governance
## Module 06: Agent Identity, RBAC, Agent Policy, Cost Controls, Guardrails, and PII Redaction

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🔹 Demonstrates how **RBAC** controls what an agent can access
2. 🔮 Shows **Agent Policy** — how to restrict what the agent can do, independent of the user's role (preview)
3. 🛠️ Configures **credit budgets** to control AI costs per user
4. 🔹 Shows the **trust boundary** — what stays in Snowflake vs. what goes external
5. 🔹 Explains **Cortex AI Guardrails** — prompt injection and jailbreak protection
6. 🛠️ Reviews the **audit trail** for agent interactions
7. 🛠️ (Optional) Demonstrates **AI_REDACT** for PII detection and redaction

---

### Why AI Security Matters:

AI agents are powerful — they can query data, generate SQL, invoke external tools, and take action. Without proper governance:
- An agent could access data beyond what a user is authorized to see
- Runaway conversations could consume excessive credits
- Adversarial prompts could attempt to override safety boundaries
- Sensitive PII could surface in agent responses
- There's no audit trail of what the agent did or why

Snowflake's AI security model ensures:
- **RBAC enforcement:** The agent operates under the calling user's role — it can never see more than the user can
- **Agent Policy:** The agent can be deliberately restricted below the caller's role with a scoped, account-level policy
- **Cost controls:** Credit budgets cap spending per user per day
- **Guardrails:** Prompt injection detection and jailbreak prevention (GA, Enterprise Edition)
- **PII redaction:** AI_REDACT masks sensitive data before it reaches users or agents
- **Encryption:** All inference traffic encrypted (TLS 1.2+), cross-region uses mTLS
- **No data persistence:** Inference payloads are transient — never stored at the processing region
- **Full audit:** Every agent call is logged in Account Usage
- *Coming soon: **Data movement policies and Trust Center AI** checks will extend these controls further — watch for GA announcements post-Summit.*

> **Documentation:** [Cortex Agents Access Control](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents#access-control-requirements)

---

### Estimated Time: **15–20 minutes**

### Prerequisites:
- Module 05 complete (agent with MCP connectors configured)

## Step 1: Set Context

In [ ]:
%%sql -r SetContext
USE ROLE ACCOUNTADMIN;
USE DATABASE NEXUS_HOL;
USE WAREHOUSE NEXUS_WH;

## Step 2: 🔹 RBAC in Action — The Agent Sees Only What Your Role Allows

### Key Principle: The agent inherits the caller's role permissions

When a user calls the agent, it executes under their role. This means:
- If a role can't see a table, the agent's SQL won't return data from it
- If a role can't use a tool, the agent can't invoke it
- Different roles get different answers from the same agent

You already demonstrated this in Module 04 with `NEXUS_SI_USER` — a role that can call the agent but can't access raw tables directly.

In production, you'd use **Row Access Policies** to enforce row-level filtering. The agent's SQL would automatically be filtered by the policy — no changes to the agent or semantic view needed.

---

### 📌 Important: Agent SQL Generation Scope (April 2026 Change)

As of April 13, 2026, Cortex Agents generate SQL **directly** rather than delegating to a separate Cortex Analyst service. This means:

**What the agent CAN access:**
- Any table the **calling user's role** has SELECT access to — not just tables in the semantic view

**What still constrains it:**
- **RBAC** — if the role can't SELECT from a table, the query fails
- **Semantic view** — the agent prioritizes tables defined in the semantic view (dimensions, metrics, verified queries guide its SQL)
- **Orchestration instructions** — you can explicitly instruct the agent to only query semantic view tables

**What this means for customers:**

If a customer requires **strict scoping** (agent can ONLY access tables in the semantic view), the recommended approach is:
1. Create a dedicated role for agent users that only has SELECT on the specific tables in the semantic view
2. Do NOT grant broad schema-level SELECT to that role
3. Optionally add orchestration instructions: *"Only query tables defined in the semantic view. Do not access other tables."*

> **Release note:** [Apr 13, 2026: Improved SQL generation in Cortex Agents](https://docs.snowflake.com/en/release-notes/2026/other/2026-04-13-cortex-agents-agentic-analyst)
>
> **Why the change:** Direct SQL generation improves accuracy and reduces latency. The agent understands your semantic view better when it generates SQL itself rather than passing through an intermediary.

---

> **Documentation:** [Row Access Policies](https://docs.snowflake.com/en/user-guide/security-row-intro) | [Cortex Agents Access Control](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents#access-control-requirements) | [Multi-tenancy for Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-multi-tenancy)

## Step 2b: 🔮 Emerging Agent Governance Patterns (Preview Concepts)

⚠️ **Preview/reference only. Some concepts shown below may reflect evolving or internal governance patterns that are not yet publicly documented or generally available. Do not execute unless these capabilities are enabled in your environment.**

---

### Why This Matters

Traditional RBAC answers:

> “What can the user do?”

Emerging AI governance patterns introduce an additional question:

> “Even if the user can do it, should the AI agent be allowed to do it automatically?”

This creates a stricter operational boundary for AI-driven execution than for direct human interaction.

Today, Snowflake governance for Cortex Agents is implemented through layered controls rather than a dedicated per-agent policy object.

---

## 🔹 Current Governance Controls for Cortex Agents

The following governance capabilities are currently documented and supported:

| Governance Layer | Purpose |
|---|---|
| Cortex AI Guardrails (`AI_SETTINGS`) | Detect prompt injection and jailbreak attempts |
| AI Function Privileges | Control which Cortex AI functions a role can invoke |
| RBAC on AGENT objects | Control who can use or manage agents |
| Underlying Resource Permissions | Restrict semantic views, search services, tables, and warehouses |
| Agent Budget Constraints | Limit execution time and token consumption |

---

## 🔹 Cortex AI Guardrails

Cortex AI Guardrails help protect Cortex Agents, Snowflake Intelligence, and Cortex Code from prompt injection and jailbreak attacks.

Example:

```sql
ALTER ACCOUNT SET AI_SETTINGS = '{
  "guardrails": {
    "prompt_injection": {"enabled": true},
    "jailbreak": {"enabled": true}
  }
}';
```

Key points:

- Configured at the account level
- Enabled by `ACCOUNTADMIN`
- Applies across supported Cortex AI surfaces
- Charged per token scanned
- Enterprise Edition feature

---

## 🔹 AI Function Privileges

Snowflake supports role-based access control for individual Cortex AI functions.

Example:

```sql
GRANT AI FUNCTION COMPLETE
ON ACCOUNT
TO ROLE analyst_role;

GRANT AI FUNCTION SUMMARIZE
ON ACCOUNT
TO ROLE analyst_role;

REVOKE AI FUNCTION CLASSIFY
ON ACCOUNT
FROM ROLE analyst_role;
```

This allows organizations to expose only approved AI capabilities to specific roles.

---

## 🔹 RBAC on AGENT Objects

Cortex Agents are schema-level Snowflake objects and participate in standard RBAC.

Example:

```sql
GRANT CREATE AGENT
ON SCHEMA NEXUS_HOL.AGENTS
TO ROLE ai_engineer;

GRANT USAGE
ON AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT
TO ROLE analyst_role;
```

This governs:
- who can create agents
- who can invoke agents
- who can manage agent versions

---

## 🔹 Agent Budget Constraints

Cortex Agents can define orchestration budgets that constrain execution cost and runtime.

Example agent configuration:

```yaml
orchestration:
  budget:
    seconds: 45
    tokens: 16000
```

This is not a security boundary, but it limits:
- execution duration
- token consumption
- runaway orchestration behavior

---

## 🔹 Agent Identity Context (Reference Concept)

Snowflake exposes session context through `SYS_CONTEXT`, enabling governance-aware patterns that distinguish AI execution from direct human activity.

> 📌 Session context fields and AI-specific identifiers may evolve as Cortex Agent governance capabilities mature.

Reference example:

```sql
SELECT
  SYS_CONTEXT('SNOWFLAKE$SESSION', 'IS_AGENT_ACTIVATED') AS is_agent,
  SYS_CONTEXT('SNOWFLAKE$SESSION', 'AGENT_TYPE')         AS agent_type,
  SYS_CONTEXT('SNOWFLAKE$SESSION', 'AGENT_NAME')         AS agent_name;
```

This enables governance logic that can behave differently for AI agents versus direct user access.

---

## 🔹 Agent-Aware Row Access Policy Pattern (Reference Concept)

`SYS_CONTEXT` can be incorporated into Row Access Policies to create stricter controls for AI-driven sessions.

Reference example:

```sql
CREATE OR REPLACE ROW ACCESS POLICY
  NEXUS_HOL.COMPLIANCE.NEXUS_AGENT_RESTRICTION
AS (status VARCHAR) RETURNS BOOLEAN ->
  CASE
    WHEN SYS_CONTEXT(
      'SNOWFLAKE$SESSION',
      'IS_AGENT_ACTIVATED'
    ) = 'TRUE'
      THEN status != 'RESTRICTED'
    ELSE TRUE
  END;
```

In this pattern:

- Human users retain normal role-based access
- AI-driven sessions can be configured to receive a more restrictive data view
- Governance is enforced directly inside Snowflake
- RBAC, RLS, masking, and AI Guardrails work together

---

## 🔹 Layered Governance Model

Effective Cortex Agent governance is typically implemented through the intersection of multiple controls:

| Layer | Enforced By | Purpose |
|---|---|---|
| RBAC / RLS | Snowflake roles and row access policies | Limits what users can access |
| AI Function Privileges | Account-level grants | Restricts which AI functions roles may invoke |
| AI Guardrails | `AI_SETTINGS` | Detects prompt injection and jailbreak attempts |
| Masking / AI_REDACT | Data-layer policies | Prevents sensitive values from reaching AI systems |
| Resource Permissions | Standard Snowflake RBAC | Limits accessible semantic views, search services, warehouses, and connectors |
| Budget Constraints | Agent orchestration config | Restricts runtime and token consumption |

The effective capabilities available to a Cortex Agent are the intersection of all configured controls.

> 📌 **Important:**  
> Snowflake does not currently expose a dedicated per-agent governance policy object such as `AGENT_SCOPE_POLICY`.
>
> In practice, agent-specific restrictions are implemented through:
> - RBAC and role design
> - ownership of the agent
> - grants on semantic views and search services
> - warehouse/database/schema access
> - approved integrations and MCP connectors
> - AI function privileges
> - orchestration budget limits
>
> Governance for Cortex Agents is therefore implemented as a layered security model rather than a single policy object.

| Status                        | Examples                                                                 |
| ----------------------------- | ------------------------------------------------------------------------ |
| GA / Documented               | AI Guardrails, AI Function Privileges, AGENT RBAC, orchestration budgets |
| Preview / Conceptual Patterns | AI-aware `SYS_CONTEXT` governance patterns                               |


## Step 3: 🛠️ Credit Budgets — Prevent Runaway AI Costs

### Control AI spending at multiple levels

Snowflake provides multiple levers to control AI costs:

| Control | Scope | How It Works |
|---------|-------|-------------|
| **Resource Monitors** | Account/Warehouse | Set credit thresholds with alerts and auto-suspend — applies to all compute including AI services |
| **CoCo daily limits** | Per-user | Caps daily CoCo CLI usage per user |
| **Orchestration budget** | Per-agent | `budget.seconds` and `budget.tokens` in the agent spec limit reasoning per request (you set this in Module 03) |
| **Warehouse auto-suspend** | Per-warehouse | Stops compute charges when idle (you set 60s in Module 01) |

The orchestration budget you already configured in the agent spec is your first line of defense:

```yaml
orchestration:
  budget:
    seconds: 45      # Agent stops reasoning after 45 seconds
    tokens: 16000    # Agent uses max 16K tokens per request
```

For account-level monitoring, use Resource Monitors and Account Usage views.

> **Documentation:** [Resource Monitors](https://docs.snowflake.com/en/user-guide/resource-monitors) | [AI Services Billing](https://docs.snowflake.com/en/user-guide/cost-understanding-compute#ai-services-costs)

In [ ]:
%%sql -r CortexBudget
-- Check current AI credit consumption (last 7 days)
SELECT
    USAGE_DATE,
    SERVICE_TYPE,
    SUM(CREDITS_USED) AS TOTAL_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND USAGE_DATE >= DATEADD('day', -7, CURRENT_DATE())
GROUP BY USAGE_DATE, SERVICE_TYPE
ORDER BY USAGE_DATE DESC;

## Step 4: 🔹 Cortex AI Guardrails

### Prompt Injection & Jailbreak Protection

**Cortex AI Guardrails** (GA, Enterprise Edition) provide run-time protection against adversarial prompts:

| Protection | What It Detects |
|------------|----------------|
| **Prompt injection** | Attempts to override system instructions through malicious prompts, including indirect injections in tool calls |
| **Jailbreak prevention** | Attempts to bypass the model's safety protocols and security boundaries |
| **Zero-day protection** | Sophisticated, previously unknown attack patterns identified in real time |

Guardrails are configured at the **account level** and currently apply to **CoCo**. They use contextual reasoning to detect and neutralize malicious intent.

---

### A Note on Guardrails vs. Foundation Models

Foundation models (Claude 4, GPT-4.1, etc.) have significantly improved their built-in safety alignment. For many standard enterprise use cases, the model's native safety is sufficient — it won't generate harmful content, won't reveal system prompts, and won't execute obviously malicious instructions.

**When you still need Guardrails:**
- High-security environments where adversarial users are expected
- External-facing agents (customers/partners have access, not just internal employees)
- Compliance requirements that mandate layered defense (defense in depth)
- When the cost of a single bypass is very high (financial, legal, reputational)

**When foundation model safety may be sufficient:**
- Internal-only agents used by trusted employees
- Low-risk analytical use cases (read-only data, no external actions)
- When you've already scoped the agent narrowly via RBAC and tool restrictions

> **Documentation:** [Cortex AI Guardrails](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-ai-guardrails)

In [ ]:
%%sql -r AISettings
-- View current guardrail settings
SHOW PARAMETERS LIKE 'AI_SETTINGS' IN ACCOUNT;

In [ ]:
%%sql -r AIGuardrails
-- Enable Cortex AI Guardrails (Enterprise Edition required)
-- ╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
-- ║  XXX: Replace with the guardrail type to enable.                                                                             ║
-- ║  Hint: What kind of attack are we protecting against? Check the table in Step 4 above — it's the first protection listed.    ║
-- ╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
ALTER ACCOUNT SET AI_SETTINGS = $$
  guardrails:
    XXX:
      - enabled: true
$$;

## Step 5: 🔹 Understanding the Trust Boundary

### What Stays Inside Snowflake vs. What Goes External

```
┌─────────────────────────────────────────────────────────────┐
│                   SNOWFLAKE TRUST BOUNDARY                  │
│                                                             │
│  User Question → Agent Orchestration → Tool Selection       │
│                                                             │
│  ┌─────────────────┐    ┌──────────────────┐                │
│  │ Cortex Analyst  │    │ Cortex Search    │                │
│  │ (SQL generation)│    │ (RAG retrieval)  │                │
│  │ Data stays here │    │ Data stays here  │                │  
│  └─────────────────┘    └──────────────────┘                │
│                                                             │
│  ┌─────────────────┐                                        │
│  │ Cross-Region    │  Inference payload (transient,         │
│  │ Inference       │  encrypted, not persisted)             │
│  └────────┬────────┘                                        │
│           │                                                 │
└───────────┼─────────────────────────────────────────────────┘
            │
            │  MCP Tool Calls (external)
            ▼
┌─────────────────────┐
│  External Services  │  Only: tool call request + response
│  (Jira, Salesforce) │  Per-user OAuth, not raw data
└─────────────────────┘
```

### Key points:
- **Your data at rest never leaves your region.** Tables, stages, stored data — stays put.
- **Inference payloads** (prompts + responses) may route cross-region but are transient, encrypted, and never persisted.
- **MCP calls** send only the tool invocation (e.g., "create Jira ticket with title X") — not raw table data.
- **No egress charges** for cross-region inference.
- **Cross-region pricing:** Global ($2.00/AI credit) vs Regional ($2.20/AI credit) — determined by `CORTEX_ENABLED_CROSS_REGION`
setting.

> **Documentation:** [Cross-region Inference — Data 
Security](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference#data-security-and-residency)

| Resource | Link |
|----------|------|
| Global vs Regional AI Inference — What to Know (2.5 min video) | [Video](https://snowflake.seismic.com/Link/Content/DCpXdpMXW43Bj82MdqCbhQJq9ghG) |
| AI Pricing Improvements: Sales & GTM FAQ | [GDoc](https://docs.google.com/document/d/10k7wZLUN3tybElajcKuSccplCaYx4xEmx70HovXbVrw/edit) |
| CoCo Security FAQ | [Seismic](https://snowflake.seismic.com/Link/Content/DC3pGbjFCWG6B8F2hBCbDC2mcg8d) |

> **Key takeaway for customer conversations:** Cross-region inference unlocks newer models and better cost — but customers will
immediately ask "what happens to my data?" The video above gives you the answer before they do.

## Step 6: 🛠️ Review the Audit Trail

### Every agent interaction is logged in Account Usage
This view shows who called the agent, when, how many tokens were consumed, and the credit cost:

In [ ]:
%%sql -r AuditAgentUsage
-- View recent agent usage (who, when, tokens, cost)
SELECT START_TIME, END_TIME, USER_NAME, AGENT_NAME, TOKEN_CREDITS, TOKENS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE AGENT_NAME = 'NEXUS_AGENT'
ORDER BY START_TIME DESC
LIMIT 10;

In [ ]:
%%sql -r DailyCreditConsumption
-- Check daily credit consumption for AI services (last 7 days)
SELECT
    USAGE_DATE,
    SERVICE_TYPE,
    SUM(CREDITS_USED) AS TOTAL_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND USAGE_DATE >= DATEADD('day', -7, CURRENT_DATE())
GROUP BY USAGE_DATE, SERVICE_TYPE
ORDER BY USAGE_DATE DESC;

## Step 7: 📌 Quick Reference — How to Lock Down AI Access

| Scenario | Action |
|----------|--------|
| Disable all AI for a role | `REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE <role>` |
| Disable a specific MCP connector | `ALTER API INTEGRATION <name> SET ENABLED = FALSE` |
| Restrict agent to specific roles | Only grant `USAGE ON AGENT` to approved roles |
| Disable cross-region inference | `ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'DISABLED'` |
| Enable guardrails | `ALTER ACCOUNT SET AI_SETTINGS = $$ guardrails: ... $$` |
| Set credit budget | `ALTER ACCOUNT SET CORTEX_BUDGET_DAILY_LIMIT = <credits>` |
| Restrict agent scope below user's role | `ALTER ACCOUNT SET AGENT_POLICY = <policy>` (preview) |

---

## 🛠️ Optional: PII Redaction with AI_REDACT

### 📌 Why Masking Policies — Not Just Instructions — Are Required for PII

A common customer question: *"Can I just tell the agent not to show PII in the orchestration instructions?"*

**No. Orchestration instructions are not a security boundary.** They guide the LLM's behavior but are not guaranteed to be followed under all inputs. A sufficiently creative prompt could cause the agent to ignore instructions and surface data the SQL returned.

**The correct approach for PII protection:**

| Layer | What It Does | Bypassable by prompts? |
|-------|-------------|----------------------|
| Orchestration instructions | "Don't show PII in responses" | Yes — LLM guidance only, not enforced |
| **RBAC (role restrictions)** | User's role can't SELECT PII columns | **No** — enforced by Snowflake engine |
| **Dynamic Data Masking** | PII columns return `[REDACTED]` for non-admin roles | **No** — enforced at data layer before SQL results return |
| **AI_REDACT in masking policy** | Uses LLM to detect and mask PII in unstructured text | **No** — enforced at data layer |

**Bottom line:** If PII must never reach the LLM, enforce it at the data layer (masking policies), not in prompt instructions. The agent's SQL runs against the masked data — it never sees the raw values regardless of what the user asks.

---

### Ensure the agent never surfaces sensitive data

Even when a user has access to a table, you may want to mask PII (names, emails, phone numbers) before the data reaches the agent's response. `AI_REDACT` is a Cortex AI function that uses an LLM to detect and replace PII with category placeholders.

**Supported PII categories:** NAME, EMAIL, PHONE_NUMBER, DATE_OF_BIRTH, ADDRESS, NATIONAL_ID, PASSPORT, TAX_IDENTIFIER, PAYMENT_CARD_DATA, DRIVERS_LICENSE, IP_ADDRESS, and more.

> **Documentation:** [Detect and Redact PII](https://docs.snowflake.com/en/user-guide/snowflake-cortex/redact-pii) | [AI_REDACT Function](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)

In [ ]:
%%sql -r SeePII
-- See the raw data (PII visible)
SELECT CLIENT_NAME, EMAIL, RELATIONSHIP_MANAGER
FROM NEXUS_HOL.ANALYTICS.CLIENTS
LIMIT 5;

In [ ]:
%%sql -r ApplyAI_REDACT
-- Same data with AI_REDACT applied — PII is masked
-- ╔════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
-- ║  XXX: Replace with the PII category to redact from the EMAIL column.                                       ║
-- ║  Hint: What type of PII is an email address? Check the supported categories list in the markdown above.    ║
-- ╚════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
SELECT
    AI_REDACT(CLIENT_NAME, ['NAME']) AS CLIENT_NAME,
    AI_REDACT(EMAIL, ['XXX']) AS EMAIL,
    AI_REDACT(RELATIONSHIP_MANAGER, ['NAME']) AS RELATIONSHIP_MANAGER
FROM NEXUS_HOL.ANALYTICS.CLIENTS
LIMIT 5;

### 🔹 Production Pattern: Masking Policies + AI_REDACT

In production, you wouldn't call `AI_REDACT` manually in every query. Instead, you'd create a **Dynamic Data Masking Policy** that automatically applies redaction based on role:

```sql
-- Example: Masking policy that redacts emails for non-admin roles
CREATE OR REPLACE MASKING POLICY email_redact_policy AS (val STRING)
  RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('NEXUS_ADMIN') THEN val
    ELSE AI_REDACT(val, ['EMAIL'])
  END;

-- Apply to the EMAIL column
ALTER TABLE NEXUS_HOL.ANALYTICS.CLIENTS
  MODIFY COLUMN EMAIL SET MASKING POLICY email_redact_policy;
```

With this policy in place, when the agent generates SQL under a non-admin role, the EMAIL column automatically returns `[EMAIL]` instead of the actual address. **The agent never sees the raw PII** — the masking happens at the data layer, invisible to the agent's SQL.

> **Documentation:** [Dynamic Data Masking](https://docs.snowflake.com/en/user-guide/security-column-ddm-intro)

---

## ✅ Module 06 Complete!

### You now understand:
- How **RBAC** controls agent data access (agent inherits caller's role)
- How **Agent Policy** restricts agent session scope independently of the caller's role (preview)
- How **SYS_CONTEXT** exposes agent identity for use in governance policies
- How to set **credit budgets** for AI cost control
- How **Cortex AI Guardrails** protect against prompt injection and jailbreaks
- The **trust boundary** — what stays internal vs. what goes external
- How the **audit trail** captures all agent interactions
- How **AI_REDACT** masks PII at the data layer

---

### Key Takeaways:

1. **The agent is never more powerful than the user calling it — and with Agent Policy, it can be deliberately less powerful.** RBAC sets the ceiling. Agent Policy sets an additional restriction below that ceiling.

2. **Defense in depth.** RBAC + Agent Policy + Guardrails + PII Masking + Credit Budgets + Audit Trail = enterprise-grade AI governance.

3. **Guardrails are additive, not mandatory.** Foundation models have strong native safety. Enable guardrails when adversarial users are expected or compliance requires layered defense.

4. **Data at rest never moves.** Only inference payloads (transient, encrypted) route across regions. No egress charges.

---

### Talking Points:

- "The agent can never see more than the user can. RBAC is enforced on every query."
- "Agent Policy creates a second enforcement layer. Even if the user's role allows broad access, the agent can be restricted to read-only on a specific database. You control both what the user can do and what the agent is allowed to do on the user's behalf."
- "Credit budgets let you give teams AI access without worrying about runaway costs."
- "Every agent interaction is auditable — who asked what, what data was accessed, what actions were taken."
- "AI_REDACT ensures PII never surfaces in agent responses — even if the underlying table contains it."
- "Guardrails add prompt injection protection on top of the model's built-in safety — defense in depth for high-security environments."

---

### Next Steps:

Continue to **Module 07: CoCo** — Use CoCo to debug inefficient queries, validate agent behavior, and build pipeline steps.